# Step 0 — Sector definition and validation

## Scope
Precision agriculture applied to crop production.
Livestock and aquaculture are outside our scope.

## Initial search strategy
A patent family is selected if at least one application:
- has CPC code A01B79/005; OR
- matches both an agricultural term and a technological term
  across its title and abstract.

The keyword lists are adapted from Tey et al., Table 1.
This is an initial search strategy to be evaluated through validation.

## 1. Load the search configuration

We read the sector definition, CPC prefixes and keyword lists from
`config/sector.yaml`.

In [1]:
from pathlib import Path
import yaml

# Locate the repository from either its root or notebooks folder.
current_dir = Path.cwd()

if (current_dir / "config" / "sector.yaml").is_file():
    repo_dir = current_dir
elif (current_dir.parent / "config" / "sector.yaml").is_file():
    repo_dir = current_dir.parent
else:
    raise FileNotFoundError("Cannot find config/sector.yaml")

config_path = repo_dir / "config" / "sector.yaml"

with config_path.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

search = config["search"]

print("Configuration file:", config_path)
print("Sector:", config["sector"]["name"])
print("Scope:", config["sector"]["scope"])

for name in (
    "cpc_prefixes",
    "ipc_prefixes",
    "agricultural_terms",
    "technological_terms",
    "sector_expressions",
):
    print(f"\n{name}:")
    print(search[name])

Configuration file: /home/jovyan/team-38-PA/config/sector.yaml
Sector: Precision Agriculture
Scope: Crop production only

cpc_prefixes:
['A01B79/005']

ipc_prefixes:
[]

agricultural_terms:
['agricultur', 'farm', 'cultivat']

technological_terms:
['precis', 'smart', 'intelligen', 'digit', 'remote sens', 'global position', 'geographic inform', 'global navigation satellite', 'variable rate', 'variable-rate', 'sensor', 'drone', 'unmanned aerial vehicle', 'robot', 'auto-steer', 'auto steer', 'autonomous', 'automatic control', 'automated control', 'artificial intel', 'machine learn', 'data analytic', 'internet of things', 'internet-of-things']

sector_expressions:
[]


## 2. Connect to PATSTAT

In [2]:
from sqlalchemy import and_, or_, func

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS224_APPLN_CPC,
)

# Use TEST to check that the code works.
# Switch to PROD before drawing the actual validation sample.
#ENV = "TEST"
ENV = "PROD"

patstat = PatstatClient(env=ENV)
db = patstat.orm()

print("PATSTAT environment:", ENV)

PATSTAT environment: PROD


## 3. Define the search conditions

Our initial search has two alternative routes:

- **CPC route:** an application has a classification matching A01B79/005.
- **Keyword route:** an application contains at least one agricultural
  term AND at least one technological term across its title and abstract.

Within each keyword list, terms are connected by OR.
The two keyword lists are connected by AND.

The functions are adapted from the professor's notebook. The main change
is the separation of keywords into agricultural and technological groups.

Keyword matching uses text substrings, not whole words. This may produce
irrelevant matches, which we will investigate during validation.

In [3]:
# Our lists come from sector.yaml instead of the AI example.
AGRICULTURAL_TERMS = search["agricultural_terms"]
TECHNOLOGICAL_TERMS = search["technological_terms"]
SECTOR_CPC_PREFIXES = search["cpc_prefixes"]

if not all([
    AGRICULTURAL_TERMS,
    TECHNOLOGICAL_TERMS,
    SECTOR_CPC_PREFIXES,
]):
    raise ValueError("Agricultural, technological and CPC lists must not be empty.")


def keyword_condition(terms):
    """At least one term appears in the title or abstract."""

    # Added coalesce: treat missing titles/abstracts as empty text.
    title = func.lower(
        func.coalesce(TLS202_APPLN_TITLE.appln_title, "")
    )
    abstract = func.lower(
        func.coalesce(TLS203_APPLN_ABSTR.appln_abstract, "")
    )


    return or_(
        *(title.like(f"%{t.lower()}%") for t in terms),
        *(abstract.like(f"%{t.lower()}%") for t in terms),
    )


def symbol_prefix_condition(symbol_column, codes):
    """Match classification prefixes after removing spaces."""

    symbol = func.replace(symbol_column, " ", "")

    return or_(*(
        symbol.like(code.replace(" ", "") + "%")
        for code in codes
    ))


# Added docdb_family_id; these are still application-level rows.
APPLN_COLUMNS = (
    TLS201_APPLN.docdb_family_id,
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_auth,
    TLS201_APPLN.appln_nr,
    TLS201_APPLN.appln_kind,
    TLS201_APPLN.appln_filing_date,
)


def query_by_keywords(*columns):
    """Select applications matching both keyword groups."""

    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)

        # Outer joins allow matching when either text field is missing.
        .outerjoin(
            TLS202_APPLN_TITLE,
            TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
        )
        .outerjoin(
            TLS203_APPLN_ABSTR,
            TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
        )

        # Main change: agricultural terms AND technological terms!!!!
        # Within each group, terms are connected by OR.
        .filter(
            and_(
                keyword_condition(AGRICULTURAL_TERMS),
                keyword_condition(TECHNOLOGICAL_TERMS),
            )
        )
    )


def query_by_cpc(*columns):
    """Select applications matching our CPC prefixes."""

    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(
            TLS224_APPLN_CPC,
            TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id,
        )

        
        .filter(
            symbol_prefix_condition(
                TLS224_APPLN_CPC.cpc_class_symbol,
                SECTOR_CPC_PREFIXES,
            )
        )
    )


# IPC omitted for now because our IPC list is empty.
# Combining branches, filtering and family deduplication come next.
print("Query functions are ready.")

Query functions are ready.


## 4. Combine the search routes and identify families

We combine the CPC and keyword results using OR: either route is sufficient
for inclusion.

We retain patent-invention applications (`ipr_type = "PI"`) with a valid
DOCDB family identifier. We then select distinct family IDs using UNION.

A family is selected when at least one qualifying application matches
either search route. Families found through both routes are counted once.

In [4]:
# Select DOCDB family IDs instead of application-level details.
family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")

# Keep patent inventions with a valid family identifier.
base_filters = (
    TLS201_APPLN.docdb_family_id > 0,
    TLS201_APPLN.ipr_type == "PI",
)

families_by_cpc = (
    query_by_cpc(family_id)
    .filter(*base_filters)
)

families_by_keywords = (
    query_by_keywords(family_id)
    .filter(*base_filters)
)

# A family qualifies through CPC OR keywords.
# UNION removes duplicate family IDs, including overlaps between branches.
sector_ids = families_by_cpc.union(families_by_keywords)

# Retrieve only a small preview; no time window has been applied yet.
preview = patstat.df(
    sector_ids
    .order_by("docdb_family_id")
    .limit(10)
)

preview

,docdb_family_id
0,3478507
1,3483974
2,3487415
3,3490743
4,3545802
5,3698576
6,3762962
7,3773176
8,3774156
9,3798016


## 5. Apply the analysis period

We restrict the dataset to families dated between 2000 and 2022.

For each selected family, we calculate the minimum `earliest_filing_year`
across its patent-invention applications and apply the time filter to
that family-level value.

The end year is an initial choice intended to reduce exposure to recent
incomplete data. It does not guarantee complete coverage.

In [5]:
YEAR_MIN = config["analysis"]["year_min"]
YEAR_MAX = config["analysis"]["year_max"]
FAMILY_TIME_COLUMN = config["analysis"]["family_time_column"]

# Use the earliest available filing year across the family's
# patent-invention applications.
family_year = func.min(
    getattr(TLS201_APPLN, FAMILY_TIME_COLUMN)
)

sector_families_query = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("docdb_family_id"),
        family_year.label("family_year"),
    )
    .filter(
        TLS201_APPLN.docdb_family_id.in_(
            sector_ids.subquery().select()
        ),
        TLS201_APPLN.ipr_type == "PI",
    )
    .group_by(TLS201_APPLN.docdb_family_id)

    # Apply the time window after calculating the family-level year.
    .having(family_year.between(YEAR_MIN, YEAR_MAX))
)

preview = patstat.df(
    sector_families_query
    .order_by(TLS201_APPLN.docdb_family_id)
    .limit(10)
)

preview

,docdb_family_id,family_year
0,3828303,2001
1,3828568,2001
2,3928043,2000
3,3928427,2001
4,3947878,2001
5,3948000,2001
6,3948361,2002
7,4567102,2001
8,4575961,2000
9,4578364,2000


In [6]:
# Count the selected families without downloading the full dataset.
family_scope = sector_families_query.subquery()

count_query = (
    db.query(func.count().label("n_families"))
    .select_from(family_scope)
)

family_count = patstat.df(count_query)
family_count

,n_families
0,34448


## 6. Identify how families entered the dataset

We divide the selected families into three mutually exclusive groups:

- **CPC only:** matched the selected CPC but not the keyword conditions.
- **Keywords only:** matched the keyword conditions but not the selected CPC.
- **Both CPC and keywords:** matched both search routes.

The three counts should sum to the total candidate dataset: 34,448 families
in the current run.

This comparison shows how much each route contributes. It does not yet
measure relevance. It will help us design the validation sample and
investigate which search conditions produce irrelevant results.

At family level, the CPC match and keyword match may come from different
applications within the same family.